In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

In [2]:
# Для генерации случайных разбиений и инициализации моделей
random_seeds = [1, 42, 100, 200, 500, 1000, 2000, 3000, 4000, 5000]

In [3]:
# грузим датасет, удаляем объявления вне мкада и те, к которым внутри МКАДа не подтянулся АО
dataset = pd.read_excel('../Dataset/objects_w_geo_prepared_V0_2j.xlsx')
dataset = dataset.loc[dataset['inMKAD']==1].reset_index(drop=True)
dataset = dataset.loc[~(dataset['AO']=='НЕОПР')].reset_index(drop=True)
dataset = dataset.loc[dataset['office_class'].isin(['A', 'A+'])].reset_index(drop=True)
dataset

,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,TotalOfficeCnt_2000,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter
0,1,5.0,800,4000,A,1,8,1,0,0,...,409,0,0,313,0,0,1,ВАО,0,8554.170939
1,2,2.6,2917,7566,A,0,0,0,0,0,...,2103,1,276,703,0,0,1,САО,0,5654.790987
2,27,10.0,925,9250,A+,2,3,1,0,1,...,2281,217,120,472,0,0,1,CВАО,0,4505.076908
3,29,5.0,1990,9950,A,4,6,1,0,1,...,475,3,151,157,0,0,1,САО,0,7616.849114
4,494,15.0,2205,33080,A,4,5,1,0,1,...,1256,4,36,312,0,0,1,ЦАО,0,2542.566803
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6325,33695,24605.0,10232,251754304,A,2,79,0,0,0,...,483,570,445,175,1,0,1,ЦАО,0,5479.273526
6326,33696,46566.3,6405,298257152,A,0,0,0,0,0,...,2241,92,56,654,0,0,1,САО,0,5366.792041
6327,33697,48297.0,6405,309342272,A,32,33,0,0,0,...,2241,92,56,654,0,0,1,САО,0,5366.792041
6328,33698,48298.0,6405,309348704,A,3,33,0,0,0,...,2241,92,56,654,0,0,1,САО,0,5366.792041


In [4]:
dataset.columns

Index(['Unnamed: 0', 'space_m2', 'price_per_m2', 'total_price', 'office_class',
       'floor', 'maxfloor', 'isCU', 'isNonCU', 'isEE', 'avg_latitude',
       'avg_longtitude', 'isMall', 'isOfficeCenter', 'inTTK', 'target',
       'minDistSubway', 'cafe_500', 'cafe_1000', 'cafe_2000', 'fast_food_500',
       'fast_food_1000', 'fast_food_2000', 'restaurant_500', 'restaurant_1000',
       'restaurant_2000', 'bank_500', 'bank_1000', 'bank_2000',
       'supermarket_500', 'supermarket_1000', 'supermarket_2000',
       'clothes_500', 'clothes_1000', 'clothes_2000', 'cosmetics_500',
       'cosmetics_1000', 'cosmetics_2000', 'electronics_500',
       'electronics_1000', 'electronics_2000', 'mall_500', 'mall_1000',
       'mall_2000', 'histplaces_500', 'histplaces_1000', 'histplaces_2000',
       'coords_str', 'TotalOfficeCnt_500', 'TotalOfficeCnt_1000',
       'TotalOfficeCnt_2000', 'classACnt_500', 'classACnt_1000',
       'classACnt_2000', 'isMoscowCity', 'isSkolkovo', 'inMKAD', 'AO',
     

In [5]:
### Размечаем уникальные здания
dataset['coords_str'] = [str(round(dataset.at[i, 'avg_latitude'], 6)) + '_' + str(round(dataset.at[i, 'avg_longtitude'], 6)) for i in dataset.index]
# число уникальных точек
print('Уникальных адресов:', len(set(dataset['coords_str'])))

l1 = sorted(list(set(dataset['coords_str'])))

address_to_num = {val: i for i, val in enumerate(l1)}
dataset['address_index'] = [address_to_num[i] for i in dataset['coords_str']]
print(min(dataset.address_index), max(dataset.address_index))
dataset

Уникальных адресов: 498
0 497


,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter,address_index
0,1,5.0,800,4000,A,1,8,1,0,0,...,0,0,313,0,0,1,ВАО,0,8554.170939,466
1,2,2.6,2917,7566,A,0,0,0,0,0,...,1,276,703,0,0,1,САО,0,5654.790987,451
2,27,10.0,925,9250,A+,2,3,1,0,1,...,217,120,472,0,0,1,CВАО,0,4505.076908,435
3,29,5.0,1990,9950,A,4,6,1,0,1,...,3,151,157,0,0,1,САО,0,7616.849114,453
4,494,15.0,2205,33080,A,4,5,1,0,1,...,4,36,312,0,0,1,ЦАО,0,2542.566803,287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6325,33695,24605.0,10232,251754304,A,2,79,0,0,0,...,570,445,175,1,0,1,ЦАО,0,5479.273526,242
6326,33696,46566.3,6405,298257152,A,0,0,0,0,0,...,92,56,654,0,0,1,САО,0,5366.792041,421
6327,33697,48297.0,6405,309342272,A,32,33,0,0,0,...,92,56,654,0,0,1,САО,0,5366.792041,421
6328,33698,48298.0,6405,309348704,A,3,33,0,0,0,...,92,56,654,0,0,1,САО,0,5366.792041,421


### Строим катбуст по 10 разным разбиениям, считаем средние метрики

In [6]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [7]:
# список фичей
features = [
    # В1: Помещение + здание, 
    'floor', 'maxfloor', 
    'isCU', 'isNonCU', 'isEE', 'isOfficeCenter', 'isMall',
    'inTTK', 'inGardenRing', 'AO',
    'space_m2',
    # В2: В1 + метро
    'minDistSubway',
    # В3: В2 + окружение
    'cafe_500', 'cafe_1000', 'cafe_2000',
    'fast_food_500', 'fast_food_1000', 'fast_food_2000', 
    'restaurant_500', 'restaurant_1000', 'restaurant_2000', 
    'bank_500', 'bank_1000', 'bank_2000', 
    'supermarket_500', 'supermarket_1000', 'supermarket_2000', 
    'clothes_500', 'clothes_1000', 'clothes_2000',
    'cosmetics_500', 'cosmetics_1000', 'cosmetics_2000',
    'electronics_500', 'electronics_1000', 'electronics_2000',
    'mall_500', 'mall_1000', 'mall_2000', 
    'histplaces_500', 'histplaces_1000', 'histplaces_2000',
    'distanceToCenter',
]
cat_features = ['isCU', 'isNonCU', 'isEE', 'inTTK', 'isOfficeCenter', 'isMall', 'AO']

In [8]:
mses = []
maes = []
absolutemaes = []
r2s = []

for seed in random_seeds:
    # Создаем генератор
    rng = np.random.default_rng(seed=seed)

    # берем 110 уникальных адресов
    test_index = rng.choice(max(dataset.address_index), size=110, replace=False)
    print(test_index[0:5])
    
    # делим датасет на тестовый и проверочный
    X_train = dataset.loc[~dataset['address_index'].isin(test_index)][features]
    y_train = dataset.loc[~dataset['address_index'].isin(test_index)]['target']
    X_test = dataset.loc[dataset['address_index'].isin(test_index)][features]
    y_test = dataset.loc[dataset['address_index'].isin(test_index)]['target']
    y_testRUR = dataset.loc[dataset['address_index'].isin(test_index)]['price_per_m2']
    print(y_train.shape, y_test.shape)
    
    # создаем модель
    model = CatBoostRegressor(
        iterations=400,
        learning_rate=0.015,
        random_state=seed,
        depth=10,
        loss_function='RMSE',
        cat_features = cat_features,
        verbose=0,
        l2_leaf_reg=2.0
    )
    
    # учим  модель
    model.fit(X_train, y_train)
    
    # получаем предикты
    predictions = model.predict(X_test)
    
    # MSE, MAE
    mses.append(mean_squared_error(y_test, predictions))
    maes.append(mean_absolute_error(y_test, predictions))
    
    # AbsoluteMAE
    absolutemaes.append(mean_absolute_error(np.exp(predictions), y_testRUR))
    
    # R2
    r2s.append(model.score(X_test, y_test))

[280 401 475 383 318]
(5347,) (983,)
[193 166 374  34 392]
(4997,) (1333,)
[418 324  46 237 220]
(4313,) (2017,)
[217 300  73 284 220]
(5279,) (1051,)
[115 179 404 325  38]
(4613,) (1717,)
[264 338 194 450 277]
(4930,) (1400,)
[410 364 346 216  25]
(4898,) (1432,)
[415  27 181 467 294]
(5388,) (942,)
[227  87 457 190 129]
(5489,) (841,)
[405 315 442 108 348]
(5111,) (1219,)


In [9]:
print(f'#--- AVG MSE: {np.mean(mses):.5f}' )
print(f'#--- AVG MAE: {np.mean(maes):.5f}')
print(f'#ABS AVG MAE: {np.mean(absolutemaes):.5f}')
print(f'#--- AVG  R2: {np.mean(r2s):.5f}')

#--- AVG MSE: 0.14598
#--- AVG MAE: 0.30161
#ABS AVG MAE: 1561.00623
#--- AVG  R2: 0.32633


In [10]:
### Хар-ки офиса + признаки локации
#--- AVG MSE: 0.16244
#--- AVG MAE: 0.31974
#ABS AVG MAE: 1665.80736
#--- AVG  R2: 0.24128
### -//- + метро
#--- AVG MSE: 0.14829
#--- AVG MAE: 0.30471
#ABS AVG MAE: 1593.59523
#--- AVG  R2: 0.31195
### -//- + окружение
#--- AVG MSE: 0.14598
#--- AVG MAE: 0.30161
#ABS AVG MAE: 1561.00623
#--- AVG  R2: 0.32633

In [11]:
# Выводим значимость фичей
print("\nFeature Importances:")
dict(zip(features, model.get_feature_importance()))


Feature Importances:


{'floor': 4.462070260456353,
 'maxfloor': 4.830323365536295,
 'isCU': 2.2743763677974633,
 'isNonCU': 1.140026320596578,
 'isEE': 0.13418064522186168,
 'isOfficeCenter': 1.2454995089588372,
 'isMall': 0.0030414298675678513,
 'inTTK': 0.07559504257916157,
 'inGardenRing': 0.410798029376193,
 'AO': 4.938687084649526,
 'space_m2': 6.653987252442026,
 'minDistSubway': 7.173075551413939,
 'cafe_500': 1.6836668905667564,
 'cafe_1000': 1.4356007384371197,
 'cafe_2000': 1.7471633288704969,
 'fast_food_500': 3.2081400964551015,
 'fast_food_1000': 1.8624933071099403,
 'fast_food_2000': 2.2495659649179647,
 'restaurant_500': 3.1866161207483774,
 'restaurant_1000': 1.4661031654637482,
 'restaurant_2000': 9.019834480629529,
 'bank_500': 1.5696587916428897,
 'bank_1000': 1.3178962177570133,
 'bank_2000': 1.0572475411874418,
 'supermarket_500': 2.4123058312164374,
 'supermarket_1000': 3.144748236035099,
 'supermarket_2000': 3.31974017486595,
 'clothes_500': 1.5735688957611331,
 'clothes_1000': 1.1292

In [12]:
# Выводим топ-7 фичей по значимости для последнего разбиения
fi_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': model.get_feature_importance()
})
top_15_features = fi_df.sort_values(by='Importance', ascending=False).head(7)
display(top_15_features)

,Feature,Importance
20,restaurant_2000,9.019834
11,minDistSubway,7.173076
10,space_m2,6.653987
9,AO,4.938687
1,maxfloor,4.830323
0,floor,4.462070
37,mall_1000,3.791066


### Строим линейную модель из sklearn

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [14]:
datasetLin = dataset
datasetLin.head()

,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter,address_index
0,1,5.0,800,4000,A,1,8,1,0,0,...,0,0,313,0,0,1,ВАО,0,8554.170939,466
1,2,2.6,2917,7566,A,0,0,0,0,0,...,1,276,703,0,0,1,САО,0,5654.790987,451
2,27,10.0,925,9250,A+,2,3,1,0,1,...,217,120,472,0,0,1,CВАО,0,4505.076908,435
3,29,5.0,1990,9950,A,4,6,1,0,1,...,3,151,157,0,0,1,САО,0,7616.849114,453
4,494,15.0,2205,33080,A,4,5,1,0,1,...,4,36,312,0,0,1,ЦАО,0,2542.566803,287


In [15]:
datasetLin['nl_total_price'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['total_price'])]
datasetLin['nl_floor'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['floor'])]
datasetLin['nl_space_m2'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['space_m2'])]
datasetLin['nl_minDistSubway'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['minDistSubway'])]
datasetLin['nl_maxfloor'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['maxfloor'])]
env = [
    'cafe_500', 'cafe_1000', 'cafe_2000', 'fast_food_500', 'fast_food_1000', 'fast_food_2000', 'restaurant_500',
    'restaurant_1000', 'restaurant_2000', 'bank_500', 'bank_1000', 'bank_2000', 'supermarket_500', 'supermarket_1000', 'supermarket_2000', 'clothes_500', 'clothes_1000', 'clothes_2000',
    'cosmetics_500', 'cosmetics_1000', 'cosmetics_2000', 'electronics_500', 'electronics_1000', 'electronics_2000',
    'mall_500', 'mall_1000', 'mall_2000', 'histplaces_500', 'histplaces_1000', 'histplaces_2000', 'distanceToCenter'
]
for i in env:
    datasetLin['nl_' + i] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin[i])]
datasetLin = pd.get_dummies(datasetLin, columns=['office_class'], dtype=int)
datasetLin = pd.get_dummies(datasetLin, columns=['AO'], dtype=int)

linearFeatures = [
    'nl_floor', 'nl_maxfloor', 'isCU', 'isNonCU', 'isEE', 'isOfficeCenter', 'isMall',
    'inTTK', 'inGardenRing',
    'AO_CВАО', 'AO_ВАО', 'AO_ЗАО', 'AO_САО', 'AO_СЗАО', 'AO_ЮАО', 'AO_ЮВАО', 'AO_ЮЗАО',
    'nl_space_m2',
    # + метро
    'nl_minDistSubway',
    # + окружение
    'nl_cafe_500', 'nl_cafe_1000', 'nl_cafe_2000',
    'nl_fast_food_500', 'nl_fast_food_1000', 'nl_fast_food_2000', 
    'nl_restaurant_500', 'nl_restaurant_1000', 'nl_restaurant_2000', 
    'nl_bank_500', 'nl_bank_1000', 'nl_bank_2000', 
    'nl_supermarket_500', 'nl_supermarket_1000', 'nl_supermarket_2000',
    'nl_clothes_500', 'nl_clothes_1000', 'nl_clothes_2000',
    'nl_cosmetics_500', 'nl_cosmetics_1000', 'nl_cosmetics_2000',
    'nl_electronics_500', 'nl_electronics_1000', 'nl_electronics_2000',
    'nl_mall_500', 'nl_mall_1000', 'nl_mall_2000',
    'nl_histplaces_500', 'nl_histplaces_1000', 'nl_histplaces_2000',
    'nl_distanceToCenter'
]
exceptFeatures = []#['total_price', 'floor', 'minDistSubway', 'maxfloor'] + env
for i in features:
    if i not in linearFeatures + exceptFeatures:
        print(i)

floor
maxfloor
AO
space_m2
minDistSubway
cafe_500
cafe_1000
cafe_2000
fast_food_500
fast_food_1000
fast_food_2000
restaurant_500
restaurant_1000
restaurant_2000
bank_500
bank_1000
bank_2000
supermarket_500
supermarket_1000
supermarket_2000
clothes_500
clothes_1000
clothes_2000
cosmetics_500
cosmetics_1000
cosmetics_2000
electronics_500
electronics_1000
electronics_2000
mall_500
mall_1000
mall_2000
histplaces_500
histplaces_1000
histplaces_2000
distanceToCenter


In [16]:
mses = []
maes = []
absolutemaes = []
r2s = []

for seed in random_seeds:
    # Создаем генератор
    rng = np.random.default_rng(seed=seed)

    # берем 110 уникальных адресов
    test_index = rng.choice(max(datasetLin.address_index), size=110, replace=False)
    print(test_index[0:5])
    
    X_train = datasetLin.loc[~datasetLin['address_index'].isin(test_index)][linearFeatures]
    y_train = datasetLin.loc[~datasetLin['address_index'].isin(test_index)]['target']
    X_test = datasetLin.loc[datasetLin['address_index'].isin(test_index)][linearFeatures]
    y_test = datasetLin.loc[datasetLin['address_index'].isin(test_index)]['target']
    y_testRUR = dataset.loc[datasetLin['address_index'].isin(test_index)]['price_per_m2']
    
    # учим модель
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    # считаем метрики
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    absolutemae = mean_absolute_error(np.exp(y_pred), y_testRUR)
    
    mses.append(mse)
    r2s.append(r2)
    maes.append(mae)
    absolutemaes.append(absolutemae)

[280 401 475 383 318]
[193 166 374  34 392]
[418 324  46 237 220]
[217 300  73 284 220]
[115 179 404 325  38]
[264 338 194 450 277]
[410 364 346 216  25]
[415  27 181 467 294]
[227  87 457 190 129]
[405 315 442 108 348]


In [17]:
print(f'#--- AVG MSE: {np.mean(mses):.5f}' )
print(f'#--- AVG MAE: {np.mean(maes):.5f}')
print(f'#ABS AVG MAE: {np.mean(absolutemaes):.5f}')
print(f'#--- AVG  R2: {np.mean(r2s):.5f}')

#--- AVG MSE: 0.16646
#--- AVG MAE: 0.32155
#ABS AVG MAE: 1704.44231
#--- AVG  R2: 0.23876


In [18]:
### Хар-ки офиса + признаки локации
#--- AVG MSE: 0.15067
#--- AVG MAE: 0.30501
#ABS AVG MAE: 1598.11613
#--- AVG  R2: 0.30376
### -//- + метро
#--- AVG MSE: 0.15138
#--- AVG MAE: 0.30877
#ABS AVG MAE: 1621.51860
#--- AVG  R2: 0.29900
### -//- + окружение
#--- AVG MSE: 0.16646
#--- AVG MAE: 0.32155
#ABS AVG MAE: 1704.44231
#--- AVG  R2: 0.23876

In [19]:
# коэфтцтенты
coefficients = model.coef_
intercept = model.intercept_

print(f"Intercept: {intercept}")
for i in range(len(coefficients)):
    if abs(coefficients[i]) > 0.12:
        print(round(coefficients[i], 5), linearFeatures[i])

Intercept: 10.834799661482336
-0.24787 isMall
-0.35286 AO_ВАО
0.24641 AO_ЗАО
0.4278 AO_САО
0.13085 AO_ЮАО
-0.49442 AO_ЮВАО
0.12696 AO_ЮЗАО
0.14633 nl_cafe_2000
-0.12571 nl_fast_food_2000
0.14883 nl_restaurant_2000
-0.1362 nl_bank_2000
0.13416 nl_mall_500
-0.21149 nl_mall_1000
-0.30249 nl_distanceToCenter


In [20]:
coef_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Coefficient": model.coef_,  # Use model.coef_[0] for logistic regression
    }
)

# 2. Sort by absolute value to find the strongest positive and negative impacts
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()
top = coef_df.sort_values(by="Abs_Coefficient", ascending=False)

# 3. Display the result
display(top[["Feature", "Coefficient"]])

,Feature,Coefficient
15,AO_ЮВАО,-0.494424
12,AO_САО,0.427798
10,AO_ВАО,-0.352862
49,nl_distanceToCenter,-0.302491
6,isMall,-0.247868
11,AO_ЗАО,0.246408
44,nl_mall_1000,-0.211491
27,nl_restaurant_2000,0.148830
21,nl_cafe_2000,0.146331
30,nl_bank_2000,-0.136201


In [21]:
tempdf = pd.DataFrame()
for i in ['nl_minDistSubway', 'nl_restaurant_500', 'nl_restaurant_1000', 'nl_restaurant_2000',
         'nl_supermarket_500', 'nl_supermarket_1000', 'nl_supermarket_2000',
         'nl_fast_food_500', 'nl_fast_food_1000', 'nl_fast_food_2000',
         'nl_cafe_500', 'nl_cafe_1000', 'nl_cafe_2000',
         'nl_histplaces_500', 'nl_histplaces_1000', 'nl_histplaces_2000']:
    t = top.loc[top['Feature'] == i].reset_index(drop=True)
    tempdf = pd.concat([tempdf, t])
tempdf[['Feature', 'Coefficient']]
print(tempdf[['Coefficient']].to_string(index=False))

 Coefficient
   -0.028200
    0.058915
   -0.114886
    0.148830
    0.044361
   -0.007052
   -0.073681
   -0.087043
   -0.004550
   -0.125713
    0.056130
    0.040222
    0.146331
    0.076155
   -0.047680
   -0.003421
